# lec01-dev

Data from https://github.com/jahearn3/commuting/blob/master/commuting_port_orchard_driving.csv.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")

# DSC 80 preferred styles
pio.templates["dsc80"] = go.layout.Template(
    layout=dict(
        margin=dict(l=30, r=30, t=30, b=30),
        autosize=True,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True),
        title=dict(x=0.5, xanchor="center"),
    )
)
pio.templates.default = "simple_white+dsc80"

import warnings
warnings.simplefilter('ignore')

import datetime

In [8]:
def process_data(filename='commuting_data.csv'):
    '''Import the data from a csv file and perform initial processing tasks.
    
    Args:
        filename (csv file, optional): The csv file with the data.
    
    Returns
        pandas DataFrame
    '''

    df = pd.read_csv(filename)

    if('ferry' in filename):
        time_columns = ['home_departure_time', 'work_departure_time', 'work_arrival_time', 'home_arrival_time', 'park_in_line_southworth', 'park_on_southworth_ferry_time', 'southworth_ferry_launch_time', 'fauntleroy_ferry_departure_time']
    else:
        time_columns = ['home_departure_time', 'work_departure_time', 'work_arrival_time', 'home_arrival_time']

    # store times as datetime types
    for ts in time_columns:
        if ts in df.columns:
            df[ts] = pd.to_datetime(df['date'] + ' ' + df[ts], errors='coerce')
    # df['home_departure_time'] = pd.to_datetime(df['date'] + ' ' + df['home_departure_time'], errors='coerce')
    # df['work_departure_time'] = pd.to_datetime(df['date'] + ' ' + df['work_departure_time'], errors='coerce')
    # df['work_arrival_time'] = pd.to_datetime(df['date'] + ' ' + df['work_arrival_time'], errors='coerce')
    # df['home_arrival_time'] = pd.to_datetime(df['date'] + ' ' + df['home_arrival_time'], errors='coerce')

    # calculate minutes after midnight for departure and arrival times and store these as new columns
    midnight = datetime.time()

    # Adding columns: calculate commuting minutes; remove the date info from departure time, preserving just the time info; calculate the mileage
    if 'work_arrival_time' in df.columns:
        df['minutes_to_work'] = (df['work_arrival_time'] - df['home_departure_time']).dt.total_seconds()/60
        df['home_departure_time_hr'] = (df['home_departure_time'] - pd.to_datetime(df['date'] + ' ' + str(midnight))).dt.total_seconds()/(60*60)
        df['mileage_to_work'] = df['work_arrival_mileage'] - df['home_departure_mileage']
    if 'home_arrival_time' in df.columns:
        df['minutes_to_home'] = (df['home_arrival_time'] - df['work_departure_time']).dt.total_seconds()/60
        df['work_departure_time_hr'] = (df['work_departure_time'] - pd.to_datetime(df['date'] + ' ' + str(midnight))).dt.total_seconds()/(60*60)
        df['mileage_to_home'] = df['home_arrival_mileage'] - df['work_departure_mileage']

    #print('Mean duration: {:.2f} minutes'.format(df['minutes_to_work'].mean()))
    #print('Median duration: {:.0f} minutes'.format(df['minutes_to_work'].median()))
    #print(df['home_departure_time'].dt.day_name())

    return df

In [9]:
df = (
    process_data('commuting_port_orchard_driving.csv')
    .dropna(subset=['minutes_to_work'])
    .rename(columns={'day_of_week': 'day',
                     'home_departure_time_hr': 'departure_hour',
                     'minutes_to_work': 'minutes'})
)

In [10]:
(
    df[['date', 'day', 'departure_hour', 'minutes']]
    .loc[[0, 1, 3, 15, 43]]
    .reset_index(drop=True)
)

,date,day,departure_hour,minutes
0,5/15/2023,Mon,10.816667,68.0
1,5/16/2023,Tue,7.750000,94.0
2,5/22/2023,Mon,8.450000,63.0
3,7/6/2023,Thu,8.033333,76.0
4,11/3/2023,Fri,7.833333,66.0


In [13]:
pio.renderers.default = 'notebook'

In [15]:
fig = px.scatter(df,
           x='departure_hour',
           y='minutes',
#            color='day',
           size=np.ones(len(df)) * 50,
           size_max=8)
fig.update_xaxes(title='Home Departure Time (AM)')
fig.update_yaxes(title='Minutes to Work')
fig.update_layout(title='Commuting Time vs. Home Departure Time')
fig.update_layout(width=700)

In [16]:
hline = px.line(x=[5, 12], y=[100, 100]).update_traces(line={'color': 'red', 'width': 4})
fline1 = go.Figure(fig.data + hline.data).update_layout(width=400, title='$H(x) = 100$')
fline1

In [17]:
hline = px.line(x=[5, 12], y=[46, 130]).update_traces(line={'color': 'red', 'width': 4})
go.Figure(fig.data + hline.data).update_layout(width=400, title='$H(x) = -14 + 12x$')

In [18]:
fig = px.histogram(df['minutes'])
fig.update_xaxes(title='Minutes').update_yaxes(title='Frequency') \
.update_layout(title='Distribution of Commuting Time')

In [19]:
fig = px.scatter(df,
           x='departure_hour',
           y='minutes',
           color='day',
           size=np.ones(len(df)) * 50,
           size_max=8)
fig.update_xaxes(title='Home Departure Time (AM)')
fig.update_yaxes(title='Minutes')
fig.update_layout(title='Commuting Time vs. Home Departure Time')
fig.update_layout(width=700)

In [ ]:
vals = np.array([72, 90, 61, 85, 92])

In [ ]:
def R(h):
    return 0.2 * sum([(val - h) ** 2 for val in vals])

In [ ]:
R(100)

In [ ]:
h = np.linspace(np.min(vals) - 5, np.max(vals) + 12, 100)
Rh = R(h)

In [21]:
px.line(x=h, y=Rh).update_traces(line_color='purple', line_width=4) \
.update_xaxes(title=r'$h$').update_yaxes(title=r'$R_\text{sq}(h)$') \
.update_layout(width=600, height=350,
               title=r'$R_\text{sq}(h) = \frac{1}{5} \left( (72-h)^2 + (90 - h)^2 + (61 - h)^2 + (85 - h)^2 + (92 - h)^2 \right)$')

NameError: name 'h' is not defined